Fraud Detection Model

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [4]:
# Function to create synthetic transaction data
def create_transaction_data(num_transactions: int = 2000) -> pd.DataFrame:
    np.random.seed(7)
    
    #random generted data 
    transactions = {
        'charge_amount': np.random.exponential(120, num_transactions),
        'transaction_hour': np.random.randint(0, 24, num_transactions),
        'client_age': np.random.randint(22, 63, num_transactions),
        'available_balance': np.random.normal(1000, 2000, num_transactions),
        'past_transactions': np.random.poisson(3, num_transactions),
        'foreign_transaction': np.random.randint(0, 2, num_transactions),
    }

    df = pd.DataFrame(transactions)

    # Label transactions as fraud based on high amount, low balance, few previous transactions,
    # and foreign location
    df['fraud'] = (
        (df['charge_amount'] > 300) &        
        (df['available_balance'] < 1000) &  
        (df['past_transactions'] < 4) &      
        (df['foreign_transaction'] == 1)
    ).astype(int)

    # Introduce randomness: flip fraud label for 7% of transactions to make it more realistic
    num_noisy = int(0.07 * num_transactions)
    random_rows = np.random.choice(df.index, size=num_noisy, replace=False)
    df.loc[random_rows, 'fraud'] = 1 - df.loc[random_rows, 'fraud']

    return df

#train a model to detect fraud=1, legitimate=0
def train_fraud_model(df: pd.DataFrame):
    
    #sepertae feature and target
    X = df.drop('fraud', axis=1)
    y = df['fraud']
    
    #split data 70% for training and 30% for testing
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    #scale data to not ignore small values get mean = 0 and sd= 1
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    #create decision trees to evaluate which fraud or not and learn
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced'
    )
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)

    #evaluate the trained data 
    print("\n=== Fraud Detection Model Results ===")
    #gives us overall accuracy of moedl
    print("Accuracy:", accuracy_score(y_test, y_pred))
    #see if model predicted correctly or no if legitimate or fraud
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
    #will also evaluate how many predicitons were correct
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

    return model, scaler

#function to Predict fraud on new transactions
def predict_fraud(model, scaler, new_data: pd.DataFrame):
    X_new = scaler.transform(new_data)
    predictions = model.predict(X_new)
    return predictions

if __name__ == "__main__":
    # Generate synthetic transaction data
    df_transactions = create_transaction_data(2000)

    # Train the fraud detection model
    model, scaler = train_fraud_model(df_transactions)

    # Example new transactions to predict
    new_transactions = pd.DataFrame({
        'charge_amount': [500, 50, 3000],          
        'transaction_hour': [10, 14, 3],
        'client_age': [30, 45, 25],
        'available_balance': [1000, 5000, 200],   
        'past_transactions': [1, 10, 0],          
        'foreign_transaction': [1, 0, 1],         
    })

    # Predict fraud on new transactions
    predictions = predict_fraud(model, scaler, new_transactions)
    print("\nNew Transactions Predictions (1=Fraud, 0=Legitimate):", predictions)


=== Fraud Detection Model Results ===
Accuracy: 0.92

Confusion Matrix:
 [[551   0]
 [ 48   1]]

Classification Report:
               precision    recall  f1-score   support

           0       0.92      1.00      0.96       551
           1       1.00      0.02      0.04        49

    accuracy                           0.92       600
   macro avg       0.96      0.51      0.50       600
weighted avg       0.93      0.92      0.88       600


New Transactions Predictions (1=Fraud, 0=Legitimate): [0 0 0]
